In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage,AIMessage
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
load_dotenv()

In [3]:
llm=ChatOpenAI(model="gpt-4o-mini")

In [4]:
# Tools
import requests


search_tool=DuckDuckGoSearchRun(region="us-en")

@tool
def calculator(a:float,b:float,operation:str)->dict:
    """
    Perform a basic arithmetic operation of two numbers.
    Supported opeartions are : add , sub , mul , div
    """
    if operation == "add":
        result=a+b
    elif operation == "sub":
        result=a-b
    elif operation == "mul":
        result=a*b
    elif operation == "div":
        if(b==0):
            return {"error":"Not allowed"}
        result=a/b
    else:
        return {"error": f"Unsupported Operation {operation}"}
    return {"result":result}

@tool
def get_stock_price(symbol:str)->dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL','TSLA')
    using Alpha Vantage api key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=9E2OTGJ1SW6VEM8B"

    r=requests.get(url)
    return r.json()

In [5]:
tools=[search_tool,calculator,get_stock_price]
llm_with_tools=llm.bind_tools(tools)

In [6]:
# state

from typing import Annotated

class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [7]:
# graph nodes

def chat_node(state:ChatState):
    messages=state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node=ToolNode(tools)

In [8]:
graph=StateGraph(ChatState)
graph.add_node("chat_node",chat_node)
graph.add_node("tools",tool_node)

graph.add_edge(START,"chat_node")
graph.add_conditional_edges("chat_node",tools_condition)
graph.add_edge("tools","chat_node")

workflow=graph.compile()
workflow

In [11]:
response = workflow.invoke(
    {"messages": [HumanMessage(content="Check stock price of AAPL and calculate 100*5")]}
)
print(response["messages"][-1].content)

